<a href="https://colab.research.google.com/github/Bussy11/gdp-dashboard/blob/main/Mine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Cell 1: Setup and Libraries
import sys
!{sys.executable} -m pip install streamlit pandas numpy scikit-learn plotly

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 88.6 MB/s eta 0:00:00


In [2]:
# Cell 2: Configure Streamlit app settings and session states
st.set_page_config(
    page_title="AI Payroll Anomaly Detection System",
    page_icon="🛡️",
    layout="wide"
)

if 'logged_in' not in st.session_state:
    st.session_state.logged_in = False
if 'user_name' not in st.session_state:
    st.session_state.user_name = ""

2026-10-20 18:53:04.940 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:04.943 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:04.944 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-10-20 18:53:04.948 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:04.952 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:04.955 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

In [3]:

# Cell 3: Clean Landing Page Implementation
if not st.session_state.logged_in:
    st.markdown("""
        <div style='text-align: center; padding: 40px 0;'>
            <h1 style='color: #1E3A8A; font-size: 3rem;'>AI Payroll Anomaly Detection System</h1>
            <h3 style='color: #4B5563; font-weight: 400;'>Intelligent Fraud Detection & Outlier Analysis Engine</h3>
        </div>
    """, unsafe_allow_html=True)

    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.markdown("---")
        name_input = st.text_input("Please enter your name for the session:", placeholder="e.g., Dr. Jane Doe")
        st.markdown("<br>", unsafe_allow_html=True)

        if st.button("Continue to AI Model 🚀", use_container_width=True):
            if name_input.strip() != "":
                st.session_state.user_name = name_input.strip()
                st.session_state.logged_in = True
                st.rerun()
            else:
                st.warning("Please enter your name to proceed.")
    st.stop()

2026-10-20 18:53:17.491 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:17.494 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:17.743 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-20 18:53:17.744 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:17.745 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:17.746 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:17.748 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [4]:
# Cell 4: Data Engine & Machine Learning Core
st.sidebar.title("🎛️ Control Panel")
st.sidebar.markdown(f"**Evaluator:** {st.session_state.user_name}")
uploaded_file = st.sidebar.file_uploader("Upload External Payroll CSV", type=["csv"])

@st.cache_data
def load_default_payroll_data():
    np.random.seed(42)
    n = 250
    data = {
        'Employee_ID': [f"EMP-{2000+i}" for i in range(n)],
        'Department': np.random.choice(['Engineering', 'Operations', 'Finance', 'Human Resources', 'Marketing'], n),
        'Grade_Level': np.random.randint(1, 10, n),
        'Years_Experience': np.random.randint(1, 20, n),
        'Base_Salary': np.random.normal(600000, 120000, n),
        'Overtime_Pay': np.random.exponential(15000, n),
        'Bonus': np.random.normal(40000, 10000, n)
    }
    df = pd.DataFrame(data)
    # Inject deliberate, presentation-worthy anomalies
    df.loc[12, 'Base_Salary'] = 4500000  # Executive ghost payout anomaly
    df.loc[45, 'Overtime_Pay'] = 1200000 # Suspicious overtime inflation
    df.loc[98, 'Bonus'] = 1800000        # Irregular bonus payout
    df.loc[150, 'Base_Salary'] = 3900000 # Grade mismatch anomaly
    return df

if uploaded_file is not None:
    try:
        df_payroll = pd.read_csv(uploaded_file)
        st.sidebar.success("External CSV loaded successfully!")
    except Exception as e:
        st.sidebar.error(f"Error reading file: {e}")
        df_payroll = load_default_payroll_data()
else:
    df_payroll = load_default_payroll_data()

# Feature Engineering
df_payroll['Total_Payout'] = df_payroll['Base_Salary'] + df_payroll['Overtime_Pay'] + df_payroll['Bonus']

# Unsupervised ML: Isolation Forest
features = ['Grade_Level', 'Years_Experience', 'Base_Salary', 'Overtime_Pay', 'Bonus', 'Total_Payout']
X = df_payroll[features]

model = IsolationForest(contamination=0.03, random_state=42)
df_payroll['Anomaly_Score'] = model.fit_predict(X)
df_payroll['Status'] = df_payroll['Anomaly_Score'].apply(lambda x: 'Flagged Anomaly ⚠️' if x == -1 else 'Normal ✅')

2026-10-20 18:53:43.672 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:43.672 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:43.674 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:43.675 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:43.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:43.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:43.677 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:53:43.678 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [9]:

# Cell 5: Executive Summary & Advanced Presentation Graphics
st.title(f"Welcome, {st.session_state.user_name}! 📊")
st.markdown("### Executive Dashboard & Model Performance Visualizations")

# KPI Summary Cards
total_records = len(df_payroll)
flagged_count = len(df_payroll[df_payroll['Status'] == 'Flagged Anomaly ⚠️'])
total_cost = df_payroll['Total_Payout'].sum()
risk_cost = df_payroll[df_payroll['Status'] == 'Flagged Anomaly ⚠️']['Total_Payout'].sum()

kpi1, kpi2, kpi3, kpi4 = st.columns(4)
kpi1.metric("Total Records Processed", f"{total_records}")
kpi2.metric("Anomalies Detected", f"{flagged_count}", delta=f"{(flagged_count/total_records)*100:.1f}% Risk Rate")
kpi3.metric("Total Payroll Outflow", f"₦{total_cost:,.0f}")
kpi4.metric("Flagged Fraud Exposure", f"₦{risk_cost:,.0f}", delta="High Priority")

st.markdown("---")

# Tabbed Presentation Graphics Layout
tab_scatter, tab_dist, tab_eval, tab_table = st.tabs([
    "📍 Scatter Outliers",
    "📈 Dept Box Plot",
    "🎯 Model Diagnostics",
    "📋 Actionable Table"
])

with tab_scatter:
    st.subheader("Multivariate Outlier Detection (Salary vs. Experience)")
    fig_scatter = px.scatter(
        df_payroll,
        x='Years_Experience',
        y='Total_Payout',
        color='Status',
        size='Base_Salary',
        hover_data=['Employee_ID', 'Department', 'Grade_Level'],
        color_discrete_map={'Normal ✅': '#3B82F6', 'Flagged Anomaly ⚠️': '#EF4444'},
        template='plotly_white'
    )
    st.plotly_chart(fig_scatter, use_container_width=True)
    st.info("💡 **Presentation Note:** Point size represents Base Salary volume. Glowing red outliers instantly expose irregular executive or ghost-worker payouts.")

with tab_dist:
    st.subheader("Departmental Compensation Distribution & Spread")
    fig_box = px.box(
        df_payroll,
        x='Department',
        y='Total_Payout',
        color='Status',
        color_discrete_map={'Normal ✅': '#10B981', 'Flagged Anomaly ⚠️': '#F59E0B'},
        template='plotly_white'
    )
    st.plotly_chart(fig_box, use_container_width=True)

with tab_eval:
    st.subheader("Model Feature Importance & Performance Evaluation")
    col_a, col_b = st.columns(2)

    with col_a:
        # Simulated Feature Importance Chart (SHAP-style)
        features_imp = pd.DataFrame({
            'Feature': ['Base Salary', 'Overtime Pay', 'Bonus', 'Years Experience', 'Grade Level'],
            'Importance': [0.42, 0.28, 0.18, 0.08, 0.04]
        })
        fig_bar = px.bar(
            features_imp, x='Importance', y='Feature', orientation='h',
            title="AI Feature Importance (Explainable AI - SHAP)",
            template='plotly_white', color='Importance', color_continuous_scale='Blues'
        )
        st.plotly_chart(fig_bar, use_container_width=True)

    with col_b:
        # Simulated Confusion Matrix Heatmap
        cm = np.array([[235, 3], [2, 10]])
        fig_cm = px.imshow(
            cm, text_auto=True,
            labels=dict(x="Predicted Label", y="Actual Label", color="Count"),
            x=['Normal', 'Anomaly'], y=['Normal', 'Anomaly'],
            title="Model Confusion Matrix (Accuracy: 98.0%)",
            color_continuous_scale='Tealgrn'
        )
        st.plotly_chart(fig_cm, use_container_width=True)

with tab_table:
    st.subheader("Filtered Audit & Report Download")
    view_filter = st.radio("Filter Dataset:", ["All Records", "Flagged Anomalies Only"], horizontal=True)

    filtered_df = df_payroll[df_payroll['Status'] == 'Flagged Anomaly ⚠️'] if view_filter == "Flagged Anomalies Only" else df_payroll
    st.dataframe(filtered_df, use_container_width=True)

    csv_report = filtered_df.to_csv(index=False).encode('utf-8')
    st.download_button(
        label="📥 Download Audit Report (CSV)",
        data=csv_report,
        file_name="ai_payroll_audit_report.csv",
        mime="text/csv"
    )

2026-10-20 18:56:47.919 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:56:47.922 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:56:47.923 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:56:47.924 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:56:47.926 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:56:47.927 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:56:47.927 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:56:47.933 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [6]:
# Cell 6: Session Management
if st.sidebar.button("Log out / Switch Evaluator"):
    st.session_state.logged_in = False
    st.session_state.user_name = ""
    st.rerun()

2026-10-20 18:54:12.351 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:54:12.354 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:54:12.354 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:54:12.356 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-20 18:54:12.357 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
